# NFCI 图表导出版本参数：来源探针

冻结两次官方 FRED graph CSV 请求，只核对 `vintage_date` 是否生效，零市场结果/模型/账户。

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import csv,hashlib,json,subprocess,tempfile,tarfile,shlex
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/chicago_nfci_source_preflight_20261023';plan=json.loads((card/'graph_vintage_probe_plan.json').read_text())
assert plan['frozen'] and not (card/'graph_vintage_probe_result.json').exists()
rawdir=root/'data/snapshots/chicago_nfci_graph_vintage_20261023';rawdir.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
rows=[]
for i,url in enumerate(plan['fixed_urls']):
 with tempfile.TemporaryDirectory(prefix='nfci-graph-') as temp:
  raw=Path(temp)/'response';hdr=Path(temp)/'headers';start=datetime.now(timezone.utc).isoformat()
  cmd=['curl','--noproxy','*','--http1.1','--connect-timeout','10','--max-time','60','--fail-with-body','-sS','-L','-D',str(hdr),'-o',str(raw),url]
  proc=subprocess.run(cmd,capture_output=True,text=True,timeout=70);end=datetime.now(timezone.utc).isoformat()
  dest=rawdir/('nfci_2011_05_25.csv' if i==0 else 'nfci_2026_10_21.csv')
  if raw.exists():dest.write_bytes(raw.read_bytes())
  header_text=hdr.read_text(errors='replace') if hdr.exists() else ''
  safe_headers={}
  for line in header_text.splitlines():
   key,sep,value=line.partition(':')
   if sep and key.lower() in {'content-type','last-modified','content-length','date','cache-control'}:safe_headers[key]=value.strip()
  item={'url':url,'requested_at_utc':start,'received_at_utc':end,'curl_exit_code':proc.returncode,
        'curl_stderr':proc.stderr[:500],'response_headers_allowlisted':safe_headers,
        'bytes':dest.stat().st_size if dest.exists() else 0,'sha256':sha(dest) if dest.exists() else None}
  rows.append(item)
  if proc.returncode or not dest.exists() or dest.stat().st_size>plan['budget']['max_response_bytes_each']:
   break
  try:
   with dest.open(newline='',encoding='utf-8-sig') as f:
    table=list(csv.reader(f))
   head=table[0];dates=[x[0] for x in table[1:] if x]
   if len(head)!=2 or head[1] != 'NFCI' or not dates or not all(len(x)==10 and x[4]==x[7]=='-' for x in dates):raise ValueError('not expected NFCI CSV')
   item.update(csv_header=head,observation_rows=len(dates),first_observation_date=dates[0],last_observation_date=dates[-1])
  except Exception as exc:
   item['parse_error']=type(exc).__name__+': '+str(exc)[:120];break
  if i==0 and dates[-1]>'2011-05-25':
   item['vintage_parameter_ignored_or_unqualified']=True;break
result={'status':'source_graph_vintage_probe_incomplete_or_unqualified','plan_sha256':sha(card/'graph_vintage_probe_plan.json'),'attempts':rows,
 'GETs_consumed':len(rows),'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
if len(rows)==2 and all('csv_header' in x for x in rows) and rows[0]['sha256']!=rows[1]['sha256'] and rows[1]['last_observation_date']>rows[0]['last_observation_date']:
 result['status']='two_official_graph_CSV_vintage_samples_distinct_source_only'
 settings=_settings(root)
 with tempfile.TemporaryDirectory(prefix='nfci-graph-archive-') as tmp:
  archive=Path(tmp)/'source.tar.gz'
  with tarfile.open(archive,'w:gz') as tar:
   for dest in sorted(rawdir.glob('*.csv')):tar.add(dest,arcname=dest.name)
  digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
  assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
  remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
  assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
  result['nas_archive_sha256']=digest;result['nas_remote_hash_verified']=True
  result['nas_artifact']=receipt
(card/'graph_vintage_probe_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'GETs':len(rows),'tails':[x.get('last_observation_date') for x in rows]})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'source_graph_vintage_probe_incomplete_or_unqualified', 'GETs': 1, 'tails': ['2026-10-16']}


## 中文结论

固定的首个图表 CSV 请求带 `vintage_date=2011-05-25`，但返回 2911 行、观测尾到 2026-10-16；按原计划停止，第二请求未发。这证明公开图表导出不适合作为历史版本来源。原始误导性响应已作为拒绝证据备份 NAS 并定点恢复，不进入因子、模型或账户。`graph_vintage_probe_result.json` 与 `graph_probe_nas_binding.json` 保存来源身份。
